In [ ]:
!pip install -q langchain langchain-community faiss-cpu sentence-transformers transformers accelerate bitsandbytes

^C


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\gaksh\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\mpmath\\calculus\\__init__.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests
print(requests.__version__)

2.32.5


In [ ]:
import json

# Optional: only if used elsewhere
# import re

def extract_json(text):
    start = text.find("{")
    end = text.rfind("}") + 1

    if start == -1 or end == 0:
        raise ValueError("No JSON block found in model output")

    json_string = text[start:end]
    return json.loads(json_string)

In [ ]:
import torch
import os

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector Store
from langchain_community.vectorstores import FAISS

# Prompt
from langchain_core.prompts import PromptTemplate

# Transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# LLM Wrapper
from langchain_community.llms import HuggingFacePipeline

In [ ]:
#print("GPU Available:", torch.cuda.is_available())
#print("GPU Name:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving renal function test.txt to renal function test.txt
Saving liver function.txt to liver function.txt
Saving lipid profile.txt to lipid profile.txt
Saving thyroid.txt to thyroid.txt
Saving pcod.txt to pcod.txt
Saving vitamin.txt to vitamin.txt
Saving blood count.txt to blood count.txt
Saving db.txt to db.txt


In [ ]:
medical_text = ""

for filename in uploaded.keys():
    print(f"Reading: {filename}")
    with open(filename, "r", encoding="utf-8") as f:
        medical_text += f"\n\n===== {filename.upper()} =====\n\n"
        medical_text += f.read()

print("All files combined successfully.")
print("Total characters:", len(medical_text))

Reading: renal function test.txt
Reading: liver function.txt
Reading: lipid profile.txt
Reading: thyroid.txt
Reading: pcod.txt
Reading: vitamin.txt
Reading: blood count.txt
Reading: db.txt
All files combined successfully.
Total characters: 26942


In [ ]:
print(medical_text[:1000])




===== RENAL FUNCTION TEST.TXT =====

RENAL FUNCTION TEST (RFT) – RAG DATASET
SECTION 1: Overview

Renal Function Test (RFT) evaluates:

Kidney filtration capacity

Waste product elimination

Electrolyte balance

Acid–base status

Standard RFT Panel Includes:

Serum Creatinine

Blood Urea (BUN)

BUN/Creatinine Ratio

Estimated GFR (eGFR)

Uric Acid

Sodium

Potassium

Chloride

Bicarbonate

SECTION 2: Serum Creatinine

Normal Range:
Women: 0.6 – 1.1 mg/dL
Men: 0.7 – 1.3 mg/dL

Elevated Creatinine indicates reduced kidney filtration.

Mild elevation → Early kidney dysfunction
Rapid rise → Acute kidney injury

Creatinine depends on muscle mass.

SECTION 3: Blood Urea Nitrogen (BUN)

Normal Range: 7 – 20 mg/dL

Elevated BUN may indicate:

Kidney dysfunction

Dehydration

High protein intake

Low BUN may indicate:

Liver disease

Malnutrition

SECTION 4: BUN/Creatinine Ratio

Normal Ratio: 10:1 – 20:1

20:1 → Dehydration (Prerena


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

docs = text_splitter.create_documents([medical_text])

print("Total chunks created:", len(docs))

Total chunks created: 80


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

print("Embedding model loaded.")

/tmp/ipykernel_7380/1034558943.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embedding_model)

vectorstore.save_local("medical_vector_db")

print("Vector DB created and saved successfully.")

Vector DB created and saved successfully.


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

test_docs = retriever.invoke("diabetes diagnostic criteria")

for doc in test_docs:
    print("\n--- Retrieved Chunk ---\n")
    print(doc.page_content[:500])


--- Retrieved Chunk ---

Types:

Type 1 Diabetes: Autoimmune destruction of pancreatic beta cells → absolute insulin deficiency

Type 2 Diabetes: Insulin resistance with relative insulin deficiency

Prediabetes: Intermediate hyperglycemia not meeting diabetes threshold

SECTION 2: Core Diagnostic Criteria (Non-Pregnant Adults)

Diagnosis can be made using any one of the following:

--- Retrieved Chunk ---

SECTION 3: Diagnostic Confirmation Rules

Rule of Two (Asymptomatic Patients):
Diagnosis requires two abnormal results from either:

Same sample (e.g., A1C + FPG both elevated)
OR

Two separate test samples

Symptomatic Exception:
If classic symptoms (polyuria, polydipsia, unexplained weight loss) are present, a single Random Plasma Glucose ≥ 200 mg/dL confirms diagnosis.

--- Retrieved Chunk ---

No fasting required.

≥ 200 mg/dL + classic symptoms → diagnostic
Otherwise requires confirmation

SECTION 5: Screening Criteria (Adults)

Start screening at age ≥ 35 years.

Earlier scree

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model (this may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Creating pipeline...")
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,  # Increased from 200 to allow for longer JSON output
    temperature=0.2,
)

llm = HuggingFacePipeline(pipeline=pipe)

print("Mistral loaded successfully.")

Loading tokenizer...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Creating pipeline...
Mistral loaded successfully.


/tmp/ipykernel_7380/3287299413.py:26: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:
from langchain_core.prompts import PromptTemplate

hybrid_prompt = PromptTemplate(
    input_variables=["context", "labs", "analysis"],
    template = """
You are a clinical expert assistant.

You are given VERIFIED lab findings from a rule-based clinical system.

STRICT RULES:
- DO NOT change or reword the findings
- DO NOT invent new values
- Use findings exactly as given

Lab Findings:
{analysis}

Retrieved Medical Context:
{context}

TASK:
1. Identify relationships between abnormalities
2. Provide clinical interpretation
3. Assess overall risk level
4. Suggest next clinical steps (no diagnosis)

Return response in this JSON format:

{{
  "Findings": "",
  "Clinical_Interpretation": "",
  "Risk_Level": "",
  "Recommendations": ""
}}
"""
)

In [ ]:
def format_rule_output(rule_output):
    formatted = ""
    for section, values in rule_output.items():
        formatted += f"{section}:\n"
        if isinstance(values, dict):
            for k, v in values.items():
                formatted += f"  - {k}: {v}\n"
        else:
            formatted += f"  - {values}\n"
        formatted += "\n"
    return formatted


def run_hybrid_rag(labs):

    # 🔹 STEP 1: Rule-based analysis
    analysis = run_all_modules(labs)

    # 🔹 STEP 2: Format rules properly
    formatted_rules = format_rule_output(analysis)

    # 🔹 STEP 3: Better retrieval query
    query = f"Interpretation of lab abnormalities: {labs}"
    retrieved_docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # 🔹 STEP 4: NEW PROMPT (replace hybrid_prompt usage)
    final_prompt = f"""
You are a clinical expert assistant.

You are given VERIFIED lab findings from a rule-based clinical system.

STRICT RULES:
- DO NOT change or reword the findings
- DO NOT invent new values
- Use findings exactly as given

Lab Findings:
{formatted_rules}

Retrieved Medical Context:
{context}

TASK:
1. Identify relationships between abnormalities
2. Provide clinical interpretation
3. Assess overall risk level
4. Suggest next clinical steps (no diagnosis)

Return response in this JSON format:

{{
  "Findings": "",
  "Clinical_Interpretation": "",
  "Risk_Level": "",
  "Recommendations": ""
}}
"""

    # 🔹 STEP 5: LLM call
    response = llm.invoke(final_prompt)

    return response

In [ ]:
def diabetes_module_full(labs):
    result = {}

    fbs = labs.get("fbs")
    ppbs = labs.get("ppbs")
    rbs = labs.get("rbs")
    hba1c = labs.get("hba1c")
    c_peptide = labs.get("c_peptide")
    urine_ketones = labs.get("urine_ketones")

    # FBS
    if fbs is not None:
        if fbs >= 126:
            result["FBS"] = "Diabetes range"
        elif 100 <= fbs < 126:
            result["FBS"] = "Prediabetes"
        else:
            result["FBS"] = "Normal"

    # PPBS
    if ppbs is not None:
        if ppbs >= 200:
            result["PPBS"] = "Diabetes range"
        elif 140 <= ppbs < 200:
            result["PPBS"] = "Prediabetes"
        else:
            result["PPBS"] = "Normal"

    # HbA1c
    if hba1c is not None:
        if hba1c >= 6.5:
            result["HbA1c"] = "Diabetes range"
        elif 5.7 <= hba1c < 6.5:
            result["HbA1c"] = "Prediabetes"
        else:
            result["HbA1c"] = "Normal"

    # Type suggestion
    if c_peptide is not None:
        if c_peptide < 0.6:
            result["Type Suggestion"] = "Possible Type 1"
        elif c_peptide > 1.8:
            result["Type Suggestion"] = "Likely Type 2"
        else:
            result["Type Suggestion"] = "Indeterminate"

    if urine_ketones:
        result["Ketosis Risk"] = "Ketones detected — DKA risk"

    return result if result else {"status": "No abnormalities detected."}

In [ ]:
def cbc_module_advanced(labs, gender="female"):

    result = {}

    hemoglobin = labs.get("hemoglobin")
    rbc = labs.get("rbc")
    hematocrit = labs.get("hematocrit")
    mcv = labs.get("mcv")
    mch = labs.get("mch")
    mchc = labs.get("mchc")
    rdw = labs.get("rdw")

    wbc = labs.get("wbc")
    neutrophils = labs.get("neutrophils")
    lymphocytes = labs.get("lymphocytes")
    eosinophils = labs.get("eosinophils")
    platelets = labs.get("platelets")

    # Hemoglobin
    if hemoglobin is not None:
        if (gender == "male" and hemoglobin < 13) or \
           (gender == "female" and hemoglobin < 12):
            result["Hemoglobin"] = "Low — Anemia detected"
        elif hemoglobin > 17:
            result["Hemoglobin"] = "High — Polycythemia suspected"
        else:
            result["Hemoglobin"] = "Normal"

    # RBC Count
    if rbc is not None:
        if (gender == "male" and rbc < 4.5) or \
           (gender == "female" and rbc < 4.0):
            result["RBC Count"] = "Low — Possible anemia"
        elif rbc > 6.0:
            result["RBC Count"] = "High — Possible polycythemia"
        else:
            result["RBC Count"] = "Normal"

    # Hematocrit
    if hematocrit is not None:
        if (gender == "male" and hematocrit < 41) or \
           (gender == "female" and hematocrit < 36):
            result["Hematocrit"] = "Low — Anemia"
        elif hematocrit > 53:
            result["Hematocrit"] = "High — Polycythemia"
        else:
            result["Hematocrit"] = "Normal"

    # RBC Indices
    if mcv is not None:
        if mcv < 80:
            result["MCV"] = "Low — Microcytic anemia"
        elif mcv > 100:
            result["MCV"] = "High — Macrocytic anemia"
        else:
            result["MCV"] = "Normal"

    if mch is not None:
        if mch < 27:
            result["MCH"] = "Low — Hypochromia"
        else:
            result["MCH"] = "Normal"

    if mchc is not None:
        if mchc < 32:
            result["MCHC"] = "Low — Hypochromic anemia"
        else:
            result["MCHC"] = "Normal"

    if rdw is not None:
        if rdw > 15:
            result["RDW"] = "High — Anisocytosis (Iron deficiency likely)"
        else:
            result["RDW"] = "Normal"

    # WBC Count
    if wbc is not None:
        if wbc > 11000:
            result["WBC"] = "High — Leukocytosis (Infection / Inflammation)"
        elif wbc < 4000:
            result["WBC"] = "Low — Leukopenia"
        else:
            result["WBC"] = "Normal"

    # Differential Count
    if neutrophils is not None:
        if neutrophils > 70:
            result["Neutrophils"] = "High — Bacterial infection likely"
        elif neutrophils < 40:
            result["Neutrophils"] = "Low — Possible viral infection"
        else:
            result["Neutrophils"] = "Normal"

    if lymphocytes is not None:
        if lymphocytes > 40:
            result["Lymphocytes"] = "High — Viral infection likely"
        elif lymphocytes < 20:
            result["Lymphocytes"] = "Low"
        else:
            result["Lymphocytes"] = "Normal"

    if eosinophils is not None:
        if eosinophils > 6:
            result["Eosinophils"] = "High — Allergy / Parasitic infection"
        else:
            result["Eosinophils"] = "Normal"

    # Platelets
    if platelets is not None:
        if platelets < 150000:
            result["Platelets"] = "Low — Thrombocytopenia"
        elif platelets > 450000:
            result["Platelets"] = "High — Thrombocytosis"
        else:
            result["Platelets"] = "Normal"

    return result if result else {"status": "No abnormalities detected."}

In [ ]:
def thyroid_module_advanced(labs):

    result = {}

    tsh = labs.get("tsh")
    ft3 = labs.get("ft3")
    ft4 = labs.get("ft4")
    anti_tpo = labs.get("anti_tpo")
    trab = labs.get("trab")

    # TSH Interpretation
    if tsh is not None:
        if tsh > 4.0:
            result["TSH"] = "Elevated — Possible hypothyroidism"
        elif tsh < 0.4:
            result["TSH"] = "Low — Possible hyperthyroidism"
        else:
            result["TSH"] = "Normal"

    # Overt / Subclinical Patterns
    if tsh is not None and ft4 is not None:

        if tsh > 4.0 and ft4 < 0.8:
            result["Diagnosis"] = "Overt Hypothyroidism"
        elif tsh > 4.0 and 0.8 <= ft4 <= 1.8:
            result["Diagnosis"] = "Subclinical Hypothyroidism"
        elif tsh < 0.4 and ft4 > 1.8:
            result["Diagnosis"] = "Overt Hyperthyroidism"
        elif tsh < 0.4 and 0.8 <= ft4 <= 1.8:
            result["Diagnosis"] = "Subclinical Hyperthyroidism"
        elif tsh < 0.4 and ft4 < 0.8:
            result["Diagnosis"] = "Secondary (Central) Hypothyroidism"

    # FT3 specific check
    if ft3 is not None:
        if ft3 > 4.2:
            result["FT3"] = "Elevated — Possible T3 toxicosis"
        else:
            result["FT3"] = "Normal"

    # Autoimmune markers
    if anti_tpo is not None:
        if anti_tpo > 35:
            result["Anti-TPO"] = "Positive — Autoimmune thyroiditis likely"
        else:
            result["Anti-TPO"] = "Negative"

    if trab is not None:
        if trab > 1.75:
            result["TRAb"] = "Positive — Graves' disease likely"
        else:
            result["TRAb"] = "Negative"

    return result if result else {"status": "No abnormalities detected."}

In [ ]:
def lipid_module_advanced(labs):

    result = {}

    total_cholesterol = labs.get("total_cholesterol")
    ldl = labs.get("ldl")
    hdl = labs.get("hdl")
    triglycerides = labs.get("triglycerides")

    # =========================
    # Total Cholesterol
    # =========================
    if total_cholesterol is not None:
        if total_cholesterol >= 240:
            result["Total Cholesterol"] = "High"
        elif total_cholesterol >= 200:
            result["Total Cholesterol"] = "Borderline High"
        else:
            result["Total Cholesterol"] = "Desirable"

    # =========================
    # LDL Classification
    # =========================
    if ldl is not None:
        if ldl >= 190:
            result["LDL"] = "Very High"
        elif ldl >= 160:
            result["LDL"] = "High"
        elif ldl >= 130:
            result["LDL"] = "Borderline High"
        else:
            result["LDL"] = "Optimal/Near Optimal"

    # =========================
    # HDL Classification
    # =========================
    if hdl is not None:
        if hdl < 40:
            result["HDL"] = "Low"
        elif hdl >= 60:
            result["HDL"] = "Protective"
        else:
            result["HDL"] = "Normal"

    # =========================
    # Triglycerides
    # =========================
    if triglycerides is not None:
        if triglycerides >= 500:
            result["Triglycerides"] = "Very High"
        elif triglycerides >= 200:
            result["Triglycerides"] = "High"
        elif triglycerides >= 150:
            result["Triglycerides"] = "Borderline High"
        else:
            result["Triglycerides"] = "Normal"

    # =========================
    # Integrated Risk Pattern
    # =========================
    if ldl is not None and hdl is not None:
        if ldl >= 160 and hdl < 40:
            result["Integrated Risk"] = "Atherogenic lipid pattern"

    return result if result else {"status": "No abnormalities detected."}

In [ ]:
def micronutrient_module_advanced(labs):

    result = {}

    vitamin_d = labs.get("vitamin_d")
    b12 = labs.get("b12")
    folate = labs.get("folate")
    ferritin = labs.get("ferritin")
    iron = labs.get("iron")
    calcium = labs.get("calcium")
    magnesium = labs.get("magnesium")
    zinc = labs.get("zinc")

    # =========================
    # Vitamin D
    # =========================
    if vitamin_d is not None:
        if vitamin_d < 20:
            result["Vitamin D"] = "Deficient"
        elif vitamin_d < 30:
            result["Vitamin D"] = "Insufficient"
        else:
            result["Vitamin D"] = "Sufficient"

    # =========================
    # Vitamin B12
    # =========================
    if b12 is not None:
        if b12 < 200:
            result["Vitamin B12"] = "Deficient"
        elif b12 < 300:
            result["Vitamin B12"] = "Borderline"
        else:
            result["Vitamin B12"] = "Normal"

    # =========================
    # Folate
    # =========================
    if folate is not None and folate < 3:
        result["Folate"] = "Deficient"

    # =========================
    # Iron Stores
    # =========================
    if ferritin is not None and ferritin < 30:
        result["Ferritin"] = "Low iron stores"

    if iron is not None and iron < 60:
        result["Serum Iron"] = "Low"

    # =========================
    # Electrolyte Micronutrients
    # =========================
    if magnesium is not None and magnesium < 1.7:
        result["Magnesium"] = "Low"

    if zinc is not None and zinc < 70:
        result["Zinc"] = "Low"

    return result if result else {"status": "No abnormalities detected."}

In [ ]:
def pcod_module_full(labs):

    result = {}

    lh = labs.get("lh")
    fsh = labs.get("fsh")
    testosterone = labs.get("testosterone")
    dheas = labs.get("dheas")
    prolactin = labs.get("prolactin")
    amh = labs.get("amh")
    ovarian_volume = labs.get("ovarian_volume")
    follicle_count = labs.get("follicle_count")

    # LH/FSH ratio
    if lh and fsh:
        ratio = lh / fsh
        if ratio > 2:
            result["LH/FSH Ratio"] = "Elevated — PCOD suspicion"

    # Androgen excess
    if testosterone and testosterone > 70:
        result["Testosterone"] = "Elevated — Hyperandrogenism"

    if dheas and dheas > 350:
        result["DHEAS"] = "Elevated"

    # AMH
    if amh and amh > 4:
        result["AMH"] = "Elevated — Ovarian reserve high (PCOD pattern)"

    # Ultrasound criteria
    if ovarian_volume and ovarian_volume > 10:
        result["Ovarian Volume"] = "Increased — PCOD morphology"

    if follicle_count and follicle_count >= 12:
        result["Follicle Count"] = "Polycystic ovarian morphology"

    if prolactin and prolactin > 25:
        result["Prolactin"] = "Elevated — Rule out hyperprolactinemia"

    return result if result else {"status": "No abnormalities detected."}

In [ ]:
def pcos_module_advanced(labs):

    result = {}

    # --------------------
    # Hormonal Parameters
    # --------------------
    lh = labs.get("lh")                          # IU/L
    fsh = labs.get("fsh")                        # IU/L
    testosterone = labs.get("testosterone")      # ng/dL
    free_testosterone = labs.get("free_testosterone")
    dheas = labs.get("dheas")                    # µg/dL
    prolactin = labs.get("prolactin")            # ng/mL
    tsh = labs.get("tsh")                        # mIU/L

    # --------------------
    # Metabolic Parameters
    # --------------------
    fasting_glucose = labs.get("fasting_glucose")    # mg/dL
    fasting_insulin = labs.get("fasting_insulin")    # µIU/mL
    homa_ir = labs.get("homa_ir")

    # --------------------
    # LH / FSH Ratio
    # --------------------
    if lh is not None and fsh is not None and fsh != 0:
        lh_fsh_ratio = lh / fsh
        result["LH/FSH Ratio"] = round(lh_fsh_ratio, 2)

        if lh_fsh_ratio > 2:
            result["LH/FSH Interpretation"] = "Elevated — Suggestive of PCOS"
        else:
            result["LH/FSH Interpretation"] = "Normal"

    # --------------------
    # Androgens
    # --------------------
    if testosterone is not None:
        if testosterone > 60:
            result["Testosterone"] = "Elevated — Hyperandrogenism"
        else:
            result["Testosterone"] = "Normal"

    if free_testosterone is not None:
        if free_testosterone > 4.5:
            result["Free Testosterone"] = "Elevated — Hyperandrogenism"
        else:
            result["Free Testosterone"] = "Normal"

    if dheas is not None:
        if dheas > 350:
            result["DHEAS"] = "Elevated — Adrenal androgen excess possible"
        else:
            result["DHEAS"] = "Normal"

    # --------------------
    # Prolactin
    # --------------------
    if prolactin is not None:
        if prolactin > 25:
            result["Prolactin"] = "Elevated — Rule out hyperprolactinemia"
        else:
            result["Prolactin"] = "Normal"

    # --------------------
    # Thyroid (Exclusion)
    # --------------------
    if tsh is not None:
        if tsh > 4.0:
            result["TSH"] = "Elevated — Hypothyroidism may mimic PCOS"
        else:
            result["TSH"] = "Normal"

    # --------------------
    # Insulin Resistance
    # --------------------
    if homa_ir is not None:
        if homa_ir > 2.5:
            result["Insulin Resistance"] = "Present — Common in PCOS"
        else:
            result["Insulin Resistance"] = "Absent"

    elif fasting_glucose is not None and fasting_insulin is not None:
        calculated_homa = (fasting_glucose * fasting_insulin) / 405
        result["HOMA-IR (Calculated)"] = round(calculated_homa, 2)

        if calculated_homa > 2.5:
            result["Insulin Resistance"] = "Present — Common in PCOS"
        else:
            result["Insulin Resistance"] = "Absent"

    # --------------------
    # Final Pattern Recognition
    # --------------------
    pcos_flags = 0

    if lh is not None and fsh is not None and (lh / fsh) > 2:
        pcos_flags += 1
    if testosterone is not None and testosterone > 60:
        pcos_flags += 1
    if homa_ir is not None and homa_ir > 2.5:
        pcos_flags += 1

    if pcos_flags >= 2:
        result["Diagnosis"] = "PCOS / PCOD Likely (Based on Lab Pattern)"
    else:
        result["Diagnosis"] = "PCOS Not Strongly Suggested by Labs Alone"

    return result if result else {"status": "No abnormalities detected."}

In [ ]:
def liver_module_advanced(labs):

    result = {}

    # --------------------
    # Enzymes
    # --------------------
    ast = labs.get("ast")                    # U/L
    alt = labs.get("alt")                    # U/L
    alp = labs.get("alp")                    # U/L
    ggt = labs.get("ggt")                    # U/L

    # --------------------
    # Bilirubin
    # --------------------
    total_bilirubin = labs.get("total_bilirubin")    # mg/dL
    direct_bilirubin = labs.get("direct_bilirubin")  # mg/dL

    # --------------------
    # Synthetic Function
    # --------------------
    albumin = labs.get("albumin")            # g/dL
    inr = labs.get("inr")

    # --------------------
    # AST / ALT
    # --------------------
    if ast is not None:
        if ast > 40:
            result["AST"] = "Elevated — Hepatocellular injury possible"
        else:
            result["AST"] = "Normal"

    if alt is not None:
        if alt > 40:
            result["ALT"] = "Elevated — Hepatocellular injury possible"
        else:
            result["ALT"] = "Normal"

    # --------------------
    # AST / ALT Ratio
    # --------------------
    if ast is not None and alt is not None and alt != 0:
        ratio = ast / alt
        result["AST/ALT Ratio"] = round(ratio, 2)

        if ratio > 2:
            result["AST/ALT Interpretation"] = "Suggestive of alcoholic liver disease"

    # --------------------
    # ALP
    # --------------------
    if alp is not None:
        if alp > 120:
            result["ALP"] = "Elevated — Cholestasis or biliary obstruction possible"
        else:
            result["ALP"] = "Normal"

    # --------------------
    # GGT
    # --------------------
    if ggt is not None:
        if ggt > 60:
            result["GGT"] = "Elevated — Alcohol use or cholestatic injury possible"
        else:
            result["GGT"] = "Normal"

    # --------------------
    # Bilirubin
    # --------------------
    if total_bilirubin is not None:
        if total_bilirubin > 1.2:
            result["Total Bilirubin"] = "Elevated — Jaundice possible"
        else:
            result["Total Bilirubin"] = "Normal"

    if direct_bilirubin is not None:
        if direct_bilirubin > 0.3:
            result["Direct Bilirubin"] = "Elevated — Conjugated hyperbilirubinemia"
        else:
            result["Direct Bilirubin"] = "Normal"

    # --------------------
    # Albumin
    # --------------------
    if albumin is not None:
        if albumin < 3.5:
            result["Albumin"] = "Low — Reduced liver synthetic function"
        else:
            result["Albumin"] = "Normal"

    # --------------------
    # INR
    # --------------------
    if inr is not None:
        if inr > 1.2:
            result["INR"] = "Prolonged — Impaired liver synthetic function"
        else:
            result["INR"] = "Normal"

    # --------------------
    # Pattern Recognition
    # --------------------
    if ast is not None and alt is not None:
        if ast > 40 and alt > 40:
            result["Pattern"] = "Hepatocellular injury pattern"

    if alp is not None and ggt is not None:
        if alp > 120 and ggt > 60:
            result["Pattern"] = "Cholestatic injury pattern"

    if ast is not None and alt is not None:
        if ast > 2 * alt and ast > 40:
            result["Pattern"] = "Alcohol-related liver injury pattern"

    if albumin is not None and inr is not None:
        if albumin < 3.5 and inr > 1.2:
            result["Pattern"] = "Impaired liver synthetic function"

    return result if result else {"status": "No abnormalities detected."}

In [ ]:
def renal_module_advanced(labs, gender="female"):

    result = {}

    # =========================
    # Parameter Extraction
    # =========================
    creatinine = labs.get("creatinine")      # mg/dL
    urea = labs.get("urea")                  # mg/dL
    bun = labs.get("bun")                    # mg/dL
    egfr = labs.get("egfr")                  # mL/min/1.73m2

    sodium = labs.get("sodium")              # mEq/L
    potassium = labs.get("potassium")        # mEq/L
    chloride = labs.get("chloride")          # mEq/L
    bicarbonate = labs.get("bicarbonate")    # mEq/L

    calcium = labs.get("calcium")            # mg/dL
    phosphorus = labs.get("phosphorus")      # mg/dL
    albumin = labs.get("albumin")            # g/dL

    urine_albumin = labs.get("urine_albumin")  # mg/g (ACR preferred)
    urine_rbc = labs.get("urine_rbc")
    urine_casts = labs.get("urine_casts")

    # =========================
    # Creatinine Interpretation (Gender-Specific)
    # =========================
    if creatinine is not None:
        if (gender == "male" and creatinine > 1.3) or \
           (gender == "female" and creatinine > 1.1):
            result["Creatinine"] = "Elevated — Reduced renal filtration suspected"
        else:
            result["Creatinine"] = "Normal"

    # =========================
    # Urea / BUN
    # =========================
    if urea is not None:
        result["Urea"] = "Elevated" if urea > 40 else "Normal"

    if bun is not None:
        result["BUN"] = "Elevated" if bun > 20 else "Normal"

    # =========================
    # KDIGO eGFR Staging
    # =========================
    if egfr is not None:
        if egfr < 15:
            stage = "G5 — Kidney Failure"
        elif egfr < 30:
            stage = "G4 — Severe Decrease"
        elif egfr < 45:
            stage = "G3b — Moderate-Severe"
        elif egfr < 60:
            stage = "G3a — Moderate"
        elif egfr < 90:
            stage = "G2 — Mild Decrease"
        else:
            stage = "G1 — Normal"

        result["CKD Stage (eGFR)"] = stage

    # =========================
    # Electrolytes
    # =========================
    if potassium is not None:
        if potassium > 5.5:
            result["Potassium"] = "Hyperkalemia — Cardiac risk"
        elif potassium < 3.5:
            result["Potassium"] = "Hypokalemia"
        else:
            result["Potassium"] = "Normal"

    if sodium is not None:
        if sodium < 135:
            result["Sodium"] = "Hyponatremia"
        elif sodium > 145:
            result["Sodium"] = "Hypernatremia"
        else:
            result["Sodium"] = "Normal"

    if bicarbonate is not None:
        if bicarbonate < 22:
            result["Acid-Base Status"] = "Metabolic Acidosis — Renal cause possible"
        else:
            result["Acid-Base Status"] = "Normal"

    # =========================
    # CKD Mineral Bone Disorder
    # =========================
    if phosphorus is not None and phosphorus > 4.5:
        result["Phosphorus"] = "Elevated — CKD-MBD risk"

    if calcium is not None and calcium < 8.5:
        result["Calcium"] = "Low — Possible CKD mineral imbalance"

    # =========================
    # Albumin (Synthetic / Nephrotic Clue)
    # =========================
    if albumin is not None:
        if albumin < 3.5:
            result["Albumin"] = "Low — Possible nephrotic loss or chronic disease"
        else:
            result["Albumin"] = "Normal"

    # =========================
    # Urine Findings
    # =========================
    if urine_albumin is not None:
        if urine_albumin >= 300:
            result["Albuminuria Category"] = "A3 — Severe"
        elif urine_albumin >= 30:
            result["Albuminuria Category"] = "A2 — Moderate"
        else:
            result["Albuminuria Category"] = "A1 — Normal/Mild"

    if urine_rbc is not None and urine_rbc > 3:
        result["Hematuria"] = "Present — Glomerular pathology possible"

    if urine_casts is not None:
        if urine_casts.lower() == "rbc":
            result["Casts"] = "RBC Casts — Suggests glomerulonephritis"
        elif urine_casts.lower() == "granular":
            result["Casts"] = "Granular Casts — Tubular injury"

    # =========================
    # Integrated Risk Pattern
    # =========================
    if egfr is not None and urine_albumin is not None:
        if egfr < 60 and urine_albumin >= 30:
            result["Integrated Risk"] = "CKD Confirmed (Reduced eGFR + Albuminuria)"

    return result if result else {"status": "No abnormalities detected."}

In [ ]:
def format_rule_output(rule_output):
    formatted = ""
    for section, values in rule_output.items():
        formatted += f"{section}:\n"
        if isinstance(values, dict):
            for k, v in values.items():
                formatted += f"  - {k}: {v}\n"
        else:
            formatted += f"  - {values}\n"
        formatted += "\n"
    return formatted


In [ ]:
def run_hybrid_rag_v2(labs, gender="female"):

    rule_output = run_all_modules(labs, gender)

    query = f"Interpretation of lab abnormalities: {labs}"
    docs = vectorstore.similarity_search(query, k=5)

    retrieved_text = "\n\n".join([doc.page_content for doc in docs])
    formatted_rules = format_rule_output(rule_output)

    prompt_text = hybrid_prompt.format(
        context=retrieved_text,
        labs=labs,
        analysis=formatted_rules
    )

    response = llm.invoke(prompt_text)
    print("--- Raw LLM Response ---") # Added print statement for debugging
    print(response)
    print("------------------------")

    # Remove prompt echo if the LLM's invoke method echoes the prompt
    # Note: HuggingFacePipeline's invoke usually returns only the generated text,
    # so slicing might not be necessary, but keeping it for robustness if the underlying
    # pipeline still includes the prompt in its raw output before extraction.
    if response.startswith(prompt_text):
        response = response[len(prompt_text):]

    parsed_response = extract_json(response)

    return parsed_response

In [ ]:
def run_all_modules(labs, gender="female"):
    analysis_results = {}
    analysis_results["Diabetes"] = diabetes_module_full(labs)
    analysis_results["CBC"] = cbc_module_advanced(labs, gender)
    analysis_results["Thyroid"] = thyroid_module_advanced(labs)
    # Using the advanced PCOS module and adapting its output for the prompt's expected format
    pcos_analysis = pcos_module_advanced(labs)
    if "Diagnosis" in pcos_analysis:
        analysis_results["PCOD"] = {"status": pcos_analysis["Diagnosis"]}
    elif pcos_analysis:
        analysis_results["PCOD"] = {"status": "Findings present, see detailed PCOD analysis."}
    else:
        analysis_results["PCOD"] = {"status": "No abnormalities detected."}


    analysis_results["Vitamins"] = micronutrient_module_advanced(labs)
    analysis_results["Renal"] = renal_module_advanced(labs, gender)
    analysis_results["Liver"] = liver_module_advanced(labs)
    analysis_results["Lipid"] = lipid_module_advanced(labs)
    return analysis_results

In [ ]:
import json
import re

# 🔹 JSON extractor (robust version)
def extract_json(text):

    # ✅ CASE 1: Already a dictionary → return directly
    if isinstance(text, dict):
        return text

    try:
        # Extract JSON-like content
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if not match:
            print("❌ No JSON found")
            return None

        json_str = match.group()

        # Fix single quotes → double quotes
        json_str = json_str.replace("'", '"')

        # Fix broken apostrophes
        json_str = re.sub(r'(\w)"s\b', r'\1s', json_str)

        return json.loads(json_str)

    except Exception as e:
        print("❌ Still failed:", e)
        return None

In [ ]:
labs = {
    "fbs": 165,
    "creatinine": 1.6,
    "egfr": 52,
    "urine_acr": 120,
    "vitamin_d": 18,
    "b12": 240,
    "ldl": 170,
    "hdl": 35,
    "triglycerides": 210
}

# 🔹 Run model
raw_response = run_hybrid_rag_v2(labs)

print("------ RAW OUTPUT ------")
print(raw_response)

# 🔹 Convert to structured JSON
parsed_output = extract_json(raw_response)

print("\n------ FINAL STRUCTURED OUTPUT ------")
print(json.dumps(parsed_output, indent=4))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Raw LLM Response ---

You are a clinical expert assistant.

You are given VERIFIED lab findings from a rule-based clinical system.

STRICT RULES:
- DO NOT change or reword the findings
- DO NOT invent new values
- Use findings exactly as given

Lab Findings:
Diabetes:
  - FBS: Diabetes range

CBC:
  - status: No abnormalities detected.

Thyroid:
  - status: No abnormalities detected.

PCOD:
  - status: PCOS Not Strongly Suggested by Labs Alone

Vitamins:
  - Vitamin D: Deficient
  - Vitamin B12: Borderline

Renal:
  - Creatinine: Elevated — Reduced renal filtration suspected
  - CKD Stage (eGFR): G3a — Moderate

Liver:
  - status: No abnormalities detected.

Lipid:
  - LDL: High
  - HDL: Low
  - Triglycerides: High
  - Integrated Risk: Atherogenic lipid pattern



Retrieved Medical Context:
Infertility

Endometrial hyperplasia

Type 2 Diabetes

Metabolic syndrome

Dyslipidemia

Obesity

END OF PCOS HORMONE PROFILE DATASET

===== VITAMIN.TXT =====

SECTION 1: Overview

Vitamin tests